In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/calendar')

### **Drop Rescued Data Column**

In [0]:
df=df.drop("_rescued_data")

### **Checking Schema**

In [0]:
df.printSchema()

### **Fixing Date column's format**

In [0]:
df=df.withColumn('Date',regexp_replace(col("Date"),'/','-'))

In [0]:
df=df.withColumn('Date',to_date(col('Date'),'m-d-yyyy'))

### **Drop Duplicates**

In [0]:
df=df.dropDuplicates()

### **Adding DimKey**

In [0]:
df= df.withColumn('DimCalendarKey',monotonically_increasing_id() +1)

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.calendar'):
    df_silver_calendar = spark.read.table('adventure_works.silver.calendar')
    df=df.join(df_silver_calendar,'DimCalendarKey','left_anti')


In [0]:
df.write.format('delta').mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/calendar')

In [0]:
%sql
create table if not exists adventure_works.silver.calendar
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/calendar'